# IST Scenario 6 — Bleed or clot? What the bedside can and cannot tell

_A few bedside signs, two descriptive looks, one logistic regression and its check with a ROC curve and a confusion matrix. Level: first model, no statistics background needed. About 40 minutes._

**What you will do.** A stroke is either a clot blocking an artery of the brain or a bleed into
the brain, and the two need opposite treatments. Today every patient has a brain scan before
treatment. In IST, a third of the patients entered the trial, and started aspirin or heparin,
before any scan, because their doctor judged at the bedside that the stroke was a clot. You will
see how often that judgement was wrong, which bedside signs point to a bleed, and, with one
*Logistic Regression*, whether those signs together could replace the scan. A last step tests the
model on patients it has not seen, with a ROC curve and a confusion matrix.

**Why it matters.** About 8 to 9 strokes in 10 are *ischaemic*: a clot blocks an artery. The rest
are *haemorrhagic*: an artery bursts and blood spreads into the brain. Drugs that help with a clot
(aspirin, heparin, clot-dissolving drugs) can make a bleed worse. Both kinds damage the brain in
the same places and produce the same kinds of signs.

**Words you will meet.** A *CT scan* is an X-ray picture of the brain, on which fresh blood shows
as a white area. The *final diagnosis* is the type of stroke established later, from a scan or an
autopsy. The *odds* are the chance that something happens divided by the chance that it does not
(a 10% risk is odds of 10 to 90, about 0.11). An *odds ratio* says how many times higher the odds
are with a sign than without it: above 1 the sign points to a bleed, below 1 to a clot. The
*positive class* is the outcome the model counts as the event, here the haemorrhagic stroke.

**Before you start.** IST randomised 19,435 patients between 1991 and 1996. Nobody received
thrombolysis or thrombectomy, one third were randomised before a CT scan, and "dependent" was
measured by questionnaire, not by the modified Rankin Scale. The *conclusions* of the trial still
hold and are current practice; the *absolute percentages* describe 1990s stroke care and must not
be quoted to patients.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model).
No patient row ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect.

## 1. Connect to the MIP and open the IST pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("IST")      # add version="1.0" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())        # the pathology label differs: pick it from this list
    raise
ist = dm.datasets.list()[0]             # the IST pathology holds a single dataset
display(dm)
display(ist)

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    out = {}
    for item in raw:
        if isinstance(item, dict):
            out[str(item.get("code"))] = str(item.get("label"))
        else:
            out[str(item)] = str(item)
    return out


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the IST dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ist], variables=variables), filters=filters,
                        handle_missing=handler)


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == "all datasets"]


def numeric_summary(result, variables):
    """Mean, SD, median and range of numeric variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "median": d.get("q2"),
                         "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def describe_by(variables, group, base_filter=None):
    """Describe the variables separately in each level of `group` (one federated call per level).
    Returns (numeric table, categorical table) with a column per group level."""
    num, cat = [], []
    for c, label in labels_of(group).items():
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(variables, f).describe()
        n = numeric_summary(res, [v for v in variables if v.is_numerical()])
        if len(n):
            num.append(n.assign(group=label))
        for v in variables:
            if v.is_categorical():
                k = category_counts(res, v)
                if len(k):
                    cat.append(k.assign(group=label))
    num = pd.concat(num, ignore_index=True) if num else pd.DataFrame()
    cat = pd.concat(cat, ignore_index=True) if cat else pd.DataFrame()
    if len(num):
        num = num.pivot(index="variable", columns="group", values=["n", "mean", "sd"])
    if len(cat):
        cat = cat.pivot(index=["variable", "level"], columns="group", values="percent")
    return num, cat


def crosstab(exposure, outcome, base_filter=None, event="1"):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    rows = []
    for c, label in lab_x.items():
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: label, "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row["% " + lab_y.get(event, event)] = 100 * counts.get(event, 0) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: "{:.1f}"}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.2f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(5, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first level of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## Step 1. Choose the variables

`CT before randomisation` tells who had a scan before entering the trial. `Final diagnosis of
initial event` is the outcome: 1 = ischaemic stroke (a clot), 2 = haemorrhagic stroke (a bleed).
The four bedside predictors are the systolic blood pressure, the level of consciousness, atrial
fibrillation (an irregular heartbeat that can throw clots from the heart) and whether the symptoms
were noticed on waking.

| Code | Label in the MIP | Tree location (under IST) | Role | Values |
|---|---|---|---|---|
| `RCT` | CT before randomisation | Admission imaging | who had a scan before entering the trial | 0 No; 1 Yes |
| `FDIAG` | Final diagnosis of initial event | Stroke characteristics | outcome: clot (1) or bleed (2); curation-derived | 1 Ischaemic stroke; 2 Haemorrhagic stroke; 3 Stroke of unknown type; 4 Not a stroke; 9 Unknown |
| `RSBP` | Systolic blood pressure | Clinical assessment at admission / Consciousness and vitals | predictor | integer, 50-300 mmHg |
| `RCONSC` | Conscious level | Clinical assessment at admission / Consciousness and vitals | predictor | 1 Fully alert; 2 Drowsy; 3 Unconscious |
| `RATRIAL` | Atrial fibrillation | Prestroke characteristics / Cerebrovascular risk factors | predictor; empty in the pilot phase (984 patients) | 0 No; 1 Yes |
| `RSLEEP` | Symptoms noted on waking | Demographics and event / Event | predictor | 0 No; 1 Yes |

In [ ]:
rct     = dm.variables["CT before randomisation"]
fdiag   = dm.variables["Final diagnosis of initial event"]     # outcome: 1 clot, 2 bleed
rsbp    = dm.variables["Systolic blood pressure"]              # the four bedside predictors
rconsc  = dm.variables["Conscious level"]
ratrial = dm.variables["Atrial fibrillation"]
rsleep  = dm.variables["Symptoms noted on waking"]

display(mip.to_frame([rct, fdiag, rsbp, rconsc, ratrial, rsleep]))

## Step 2. How often was the bedside diagnosis wrong?

One `describe()` of the final diagnosis in each group: the patients scanned before entering the
trial, and the patients not scanned first. Like the preview of the cohort filter in the MIP
interface, it counts every patient who matches the filter; the models of Steps 4 and 6 will leave
out the patients without atrial fibrillation data.

*Expected:* with a scan first, 61 bleeds among 13,024 patients (0.5%): bleeds were kept out of the
trial. Without a scan first, 538 bleeds among 6,411 patients (8.4%), who were given aspirin,
heparin or neither by chance. Among the patients without a first scan whose stroke type was later
established (4,848 clots + 538 bleeds = 5,386), **one in ten had a bleed**.

In [ ]:
counts = {}
for level, name in (("1", "scan first"), ("0", "no scan first")):
    res = analysis([fdiag], F(rct) == level).describe()
    counts[name] = category_counts(res, fdiag).set_index("level")["count"]
counts = pd.DataFrame(counts).T.reindex(columns=list(labels_of(fdiag).values())).fillna(0)
share = counts.div(counts.sum(axis=1), axis=0).mul(100)
display(counts.astype(int))
display(share.style.format("{:.1f}"))

known = counts["Ischaemic stroke"] + counts["Haemorrhagic stroke"]
bleeds = counts["Haemorrhagic stroke"]
print(f"No scan first, type established later: {int(bleeds['no scan first'])} bleeds out of "
      f"{int(known['no scan first']):,} ({100 * bleeds['no scan first'] / known['no scan first']:.1f}%)")

ax = share.plot(kind="barh", stacked=True, figsize=(10, 2.6), color=["#4C72B0", "#C44E52", "#8C8C8C", "#CCCCCC", "#EEEEEE"])
ax.set_xlabel("% of the group")
ax.set_ylabel("")
ax.set_title("Final diagnosis, with and without a scan before entering the trial")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=5, frameon=False)
plt.tight_layout()
plt.show()

## Step 3. Do bleeds look different at the bedside?

The four bedside signs, described separately for the bleeds and the clots among the patients not
scanned first. The left chart shows the blood pressure of each group as a histogram (% of the
group), the right chart the other three signs.

*Expected:* 538 bleeds and 4,848 clots; mean systolic blood pressure 166.5 against 160.4 mmHg;
drowsy 29.0% against 15.6%; atrial fibrillation 13.5% against 20.2% (of patients with data);
symptoms noted on waking 21.7% against 29.4%. No patient was unconscious: for them a scan before
entering the trial was compulsory. On average, bleeds come with a higher blood pressure and more
drowsiness, clots more often with atrial fibrillation and on waking. But the differences are
modest: the two blood-pressure histograms sit almost on top of each other.

In [ ]:
def percent_of(result, variable, label):
    """Percentage of patients with one level of a categorical variable, from a describe result."""
    return category_counts(result, variable).set_index("level")["percent"].get(label, 0)


groups = {"bleed": "2", "clot": "1"}
colours = {"bleed": "#C44E52", "clot": "#4C72B0"}
bedside = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [3, 2]})
for name, level in groups.items():
    cohort = (F(rct) == "0") & (F(fdiag) == level)
    res = analysis([rsbp, rconsc, ratrial, rsleep], cohort).describe()
    bp = numeric_summary(res, [rsbp]).iloc[0]
    bedside[name] = {"patients": bp["n"], "mean systolic BP": bp["mean"], "% drowsy": percent_of(res, rconsc, "Drowsy"),
                     "% atrial fibrillation": percent_of(res, ratrial, "Yes"), "% woke up with it": percent_of(res, rsleep, "Yes")}
    h = analysis([rsbp], cohort, drop_missing=True).histogram(variable=rsbp, bins=10).raw["histogram"][0]
    n = np.array([c or 0 for c in h["counts"]], dtype=float)
    axes[0].bar(h["bins"][:-1], 100 * n / n.sum(), width=np.diff(h["bins"]), align="edge", alpha=0.5,
                color=colours[name], label=name)
bedside = pd.DataFrame(bedside)
display(bedside.style.format("{:.1f}"))

axes[0].set_xlabel("systolic blood pressure (mmHg)")
axes[0].set_ylabel("% of the group")
axes[0].set_title("Blood pressure: the two groups overlap")
axes[0].legend()
signs = bedside.loc[["% drowsy", "% atrial fibrillation", "% woke up with it"]]
signs.plot(kind="bar", ax=axes[1], color=[colours[c] for c in signs.columns], rot=0)
axes[1].set_ylabel("% of the group")
axes[1].set_title("The other bedside signs")
plt.tight_layout()
plt.show()

## Step 4. One model: logistic regression

In the MIP interface: filters `CT before randomisation = 0` and `Final diagnosis of initial event`
1 or 2; *Logistic Regression* with outcome `Final diagnosis of initial event`, positive class
Haemorrhagic stroke (2), predictors `Systolic blood pressure`, `Conscious level`, `Atrial
fibrillation`, `Symptoms noted on waking`. Each category is compared with its first code: fully
alert, no atrial fibrillation, symptoms not noticed on waking. Blood pressure is shown per 10 mmHg
(the odds ratio per mmHg to the power 10).

*Expected (5,131 patients and 510 bleeds: the 255 patients of Step 3 whose atrial fibrillation
was not recorded drop out), odds ratios for a bleed:* drowsy 2.45 (95% CI 1.98 to 3.02); atrial fibrillation
0.55 (0.42 to 0.72); symptoms noted on waking 0.66 (0.53 to 0.82); systolic blood pressure 1.0084
per mmHg, 1.09 per 10 mmHg (1.05 to 1.12).

Every sign makes sense. A bleed raises the pressure inside the skull and often dulls
consciousness; high blood pressure is the main cause of brain bleeds. Atrial fibrillation throws
clots from the heart, so it points to a clot. Bleeds tend to happen during activity, clots more
often during sleep.

In [ ]:
no_scan = (F(rct) == "0") & F(fdiag).isin(["1", "2"])            # not scanned first, clot or bleed
predictors = [rsbp, rconsc, ratrial, rsleep]
model = analysis([fdiag] + predictors, no_scan, drop_missing=True).logistic_regression(
    x=predictors, y=fdiag, positive_class="2")                    # the event: 2 = haemorrhagic stroke
display(model)

ors = odds_ratio_table(model, predictors)
bp_row = ors["term"].str.startswith(rsbp.label)
ors.loc[bp_row, ["odds_ratio", "ci_low", "ci_high"]] = ors.loc[bp_row, ["odds_ratio", "ci_low", "ci_high"]] ** 10
ors.loc[bp_row, "term"] = "Systolic blood pressure (per 10 mmHg)"
display(ors.style.format({"odds_ratio": "{:.2f}", "ci_low": "{:.2f}", "ci_high": "{:.2f}", "p": fmt_p}).hide(axis="index"))

# forest plot: red if the sign points to a bleed (odds ratio above 1), blue if it points to a clot
t = ors.iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 3.2))
for y, row in t.iterrows():
    colour = "#C44E52" if row["odds_ratio"] > 1 else "#4C72B0"
    ax.errorbar(row["odds_ratio"], y, xerr=[[row["odds_ratio"] - row["ci_low"]], [row["ci_high"] - row["odds_ratio"]]],
                fmt="o", capsize=5, color=colour)
ax.axvline(1, color="black", linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_xticks([0.25, 0.5, 1, 2, 4])
ax.set_xticklabels(["0.25", "0.5", "1", "2", "4"])
ax.minorticks_off()
ax.set_yticks(t.index)
ax.set_yticklabels(t["term"])
ax.set_xlabel("odds ratio (log scale): left of 1 points to a clot, right of 1 to a bleed")
ax.set_title("Bleed or clot? What each bedside sign says")
plt.tight_layout()
plt.show()

## Step 5. Could the bedside replace the scan?

The model turns the signs into a predicted risk of a bleed: add the intercept and the coefficients
of the patient's signs to obtain z; the risk is 1 / (1 + exp(-z)). Below for four patients; the red
part of each bar is the chance of a bleed, the blue part the chance of a clot.

*Expected:* 3% for an alert patient with atrial fibrillation who woke up with the stroke and has a
blood pressure of 140; 10% for an alert patient without atrial fibrillation, symptoms while awake,
160; 24% for the same patient drowsy with 180; 30% drowsy with 220. Even the patient with every
warning sign is more likely to have a clot than a bleed (7 in 10), and the patient with every
reassuring sign still has a 3% chance of a bleed. No combination of bedside signs separates the two
well enough to decide a treatment that could make a bleed worse: that is why, today, every patient
with a suspected stroke has a brain scan first.

In [ ]:
coef = model.to_frame().set_index("feature")["coefficient"]
print(coef.round(4).to_string())

patients = pd.DataFrame([
    {"patient": "alert, AF, woke up with it, BP 140", "drowsy": 0, "af": 1, "woke": 1, "sbp": 140},
    {"patient": "alert, no AF, while awake, BP 160", "drowsy": 0, "af": 0, "woke": 0, "sbp": 160},
    {"patient": "drowsy, no AF, while awake, BP 180", "drowsy": 1, "af": 0, "woke": 0, "sbp": 180},
    {"patient": "drowsy, no AF, while awake, BP 220", "drowsy": 1, "af": 0, "woke": 0, "sbp": 220},
]).set_index("patient")
z = (coef["Intercept"] + coef["RSBP"] * patients["sbp"] + coef["RCONSC[2]"] * patients["drowsy"]
     + coef["RATRIAL[1]"] * patients["af"] + coef["RSLEEP[1]"] * patients["woke"])
patients["bleed (%)"] = 100 / (1 + np.exp(-z))
patients["clot (%)"] = 100 - patients["bleed (%)"]
display(patients.style.format({"bleed (%)": "{:.1f}", "clot (%)": "{:.1f}"}))

ax = patients[["bleed (%)", "clot (%)"]].plot(kind="barh", stacked=True, figsize=(9, 3.2), color=["#C44E52", "#4C72B0"])
ax.invert_yaxis()
ax.axvline(50, color="black", linestyle="--", linewidth=1)
for y, value in enumerate(patients["bleed (%)"]):
    ax.text(value + 1, y, f"{value:.0f}% bleed", va="center", color="white", fontweight="bold")
ax.set_xlabel("% chance, predicted by the model")
ax.set_ylabel("")
ax.set_title("Even the most worrying bedside picture is more likely a clot")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False)
plt.tight_layout()
plt.show()

## Step 6. How good is the model? ROC curve and confusion matrix

In the MIP interface: same filters and roles, and switch on **Cross-validation** among the
parameters of *Logistic Regression*: it becomes *Logistic Regression Cross-validation*, with 5
folds. The platform fits the model on four fifths of the
patients and tests it on the remaining fifth, five times, so the model is always judged on patients
it has not seen. It returns two pictures.

- The **ROC curve** tries every cut-off from 0% to 100%: "call it a bleed when the predicted risk is
  above the cut-off". For each cut-off it plots the share of bleeds caught (the sensitivity, up)
  against the share of clots wrongly called bleeds (right). A perfect model hugs the top-left
  corner, with an area under the curve of 1; a coin toss follows the diagonal, with an area of 0.5.
- The **confusion matrix** counts the model's verdicts against the truth at one cut-off, which the
  platform fixes at 50%.

*Expected:* areas under the curve 0.64, 0.58, 0.64, 0.66 and 0.64 in the five folds, 0.63 on
average. Confusion matrix: 4,621 clots called clots, 510 bleeds called clots, no patient called a
bleed; accuracy 90.1%, sensitivity 0%, specificity 100%.

- The curves stay close to the diagonal: the four bedside signs separate bleeds from clots only a
  little better than chance. The prognostic model of Scenario 3, by comparison, would reach about
  0.80.
- No patient's predicted risk reaches 50%, so at the platform's cut-off the model calls everyone a
  clot. It catches no bleed at all, yet it is right 90% of the time, simply because 9 patients in 10
  had a clot. When the event is rare, accuracy misleads: look at the sensitivity and specificity.
- A lower cut-off catches more bleeds, at a price. At 10%, close to the average risk, the model
  would catch 56% of the bleeds but also flag 36% of the clots, patients who would then miss a
  helpful treatment. The ROC curve draws this trade-off for every cut-off.
- The areas vary from fold to fold because each fold holds only about 100 bleeds.

In [ ]:
cv = analysis([fdiag] + predictors, no_scan, drop_missing=True).logistic_regression_cv(
    x=predictors, y=fdiag, positive_class="2", n_splits=5)       # fit on 4/5 of the patients, test on 1/5, five times
curves = cv.raw["roc_curves"]                                   # one ROC curve per fold
cm = cv.raw["confusion_matrix"]                                 # the verdicts at the platform's 50% cut-off
print("Area under the ROC curve, per fold:", ", ".join(f"{c['auc']:.2f}" for c in curves),
      f"| average {np.mean([c['auc'] for c in curves]):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for c in curves:
    axes[0].plot(c["fpr"], c["tpr"], linewidth=1, label=f"{c['name'].replace('_', ' ')}: area {c['auc']:.2f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey", label="coin toss: area 0.50")
# each curve lists the cut-offs 0%, 1%, ..., 100%: position 10 is the 10% cut-off
caught = np.mean([c["tpr"][10] for c in curves])
flagged = np.mean([c["fpr"][10] for c in curves])
axes[0].plot(flagged, caught, "o", color="black")
axes[0].annotate(f"cut-off 10%: {caught:.0%} of bleeds caught,\n{flagged:.0%} of clots flagged",
                 (flagged, caught), xytext=(0.02, 0.85), arrowprops={"arrowstyle": "->"})
axes[0].set_xlabel("share of clots wrongly called bleeds")
axes[0].set_ylabel("share of bleeds caught")
axes[0].set_title("ROC curve, on patients the model had not seen")
axes[0].legend(fontsize=8, loc="lower right")

matrix = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])  # rows: the truth; columns: the model's verdict
axes[1].imshow(matrix, cmap="Blues")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{matrix[i, j]:,}", ha="center", va="center", fontsize=14,
                     color="white" if matrix[i, j] > matrix.max() / 2 else "black")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["called a clot", "called a bleed"])
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["was a clot", "was a bleed"])
axes[1].set_title("Confusion matrix at the platform's 50% cut-off")
plt.tight_layout()
plt.show()

n = sum(cm.values())
print(f"Accuracy {100 * (cm['tp'] + cm['tn']) / n:.1f}%, sensitivity {100 * cm['tp'] / (cm['tp'] + cm['fn']):.1f}%, "
      f"specificity {100 * cm['tn'] / (cm['tn'] + cm['fp']):.1f}%")

## The result in two sentences

> Among 5,386 IST patients who entered the trial before a scan and whose stroke type was later
> established, 1 in 10 had a bleed; a higher blood pressure (odds ratio 1.09 per 10 mmHg) and
> drowsiness (2.45) pointed to a bleed, atrial fibrillation (0.55) and waking up with the stroke
> (0.66) to a clot. Even combined, these signs gave predicted risks between about 3% and 30% and,
> tested on patients the model had not seen, an area under the ROC curve of 0.63, far too uncertain
> to replace a brain scan before treatment.

**Be careful.** These patients were those whom doctors judged to have a clot before any scan: the
most obvious bleeds, and all unconscious patients, were scanned first and are not in this group.
802 patients randomised without a scan never had their stroke type established and are left out;
the model also leaves out the patients of the pilot phase, whose atrial fibrillation was not
recorded.
These are 1990s patients: today CT and MRI are fast and available in most hospitals, and mobile
stroke units bring the scanner to the patient.

**Questions for discussion**

1. Why can the bedside not tell a bleed from a clot reliably, when both are so different on a scan?
2. 61 bleeds were found among patients scanned before entering the trial. How could a scan miss a
   bleed, or a bleed appear after the scan?
3. Imagine a remote place without a scanner. With these numbers, would you give aspirin to every
   suspected stroke, to none, or to some? What would you need to know to decide?
4. The model's odds ratios are all clearly different from 1, yet it cannot separate bleeds from
   clots. How can both be true?
5. The model is right 90% of the time. Would you let it decide who gets aspirin? What would you
   ask before trusting any "accurate" prediction model?